In [2]:
import asyncio
from pylabrobot.liquid_handling.backends.tecan.EVO_backend import EVOBackend

In [3]:
############### VARIABLES ###############

"""RoMa Variables"""
z_travel = 2000
z_tray = 125
z_scanner = None
z_carousel = None
z_sealer = None
z_centrifuge = None
z_bucket_thin = None
z_bucket_wide = None
r_plate = 900
r_carousel = 2700 #if wrong then -900
r_regrip = 0
g_tray_open = 900
g_tray_closed = 780
g_bucket_open_thin = None
g_bucket_closed_thin = None
g_bucket_open_wide = None
g_bucket_closed_wide = None

"""Location Variables"""
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

plate_1 = Point(x=7125, y=375)
#check plate coordinates below
plate_2 = Point(x=7125, y=1330)
plate_3 = Point(x=7125, y=2285)
plate_4 = Point(x=8645, y=375)
plate_5 = Point(x=8645, y=1330)
plate_6 = Point(x=8645, y=2285)
bucket_1 = Point(x=None, y=None)
bucket_2 = Point(x=None, y=None)
bucket_3 = Point(x=None, y=None)
bucket_4 = Point(x=None, y=None)
regrip = Point(x=None, y=None)
regrip_wide = Point(x=None, y=None)
carousel = Point(x=None, y=None)
sealer = Point(x=None, y=None)
scanner = Point(x=None, y=None)
centrifuge = Point(x=None, y=None)



In [4]:
############### BEGIN AND END FUNCTIONS ###############

"""""""""""""""""""""""""""""""""""""""""""""""""""
begin will establish USB connection, initialize the
FreedomEVO arms and move them to initial positions
"""""""""""""""""""""""""""""""""""""""""""""""""""
#PULNGER INITIALIZATION FOR LiHa should be implemented. TEST!!!
async def begin():
    backend = EVOBackend()
    await backend.io.setup()
    resp1 = await backend.send_command("W1", "PIA")
    print(resp1)
    resp2 = await backend.send_command("C5", "PIA")
    print(resp2)
    resp3 = await backend.send_command("C1", "PIA")
    print(resp3)
    resp4 = await backend.send_command("C5", "PID")
    print(resp4)
    await backend.send_command("C1", "PAA",[14628, 1999, 2000, 1800, 900])
    await backend.send_command("C5", "PAA", [9241, 793, 0, 1500, 1500, 1500, 1500, 1500, 1500, 1500, 1500])
    await backend.send_command("W1", "PAA", [-100, -700, 2000, 0, 280])

"""""""""""""""""""""""""""""""""""""""""""""""""""
end will return arms to initial positions and
terminate USB connection
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def end():
    await backend.send_command("C1", "PAA",[14628, 1999, 2000, 1800, 900])
    await backend.send_command("C5", "PAA", [9241, 793, 0, 1500, 1500, 1500, 1500, 1500, 1500, 1500, 1500])
    await backend.send_command("W1", "PAA", [-100, -700, 2000, 0, 280])
    await backend.stop

In [5]:
############### ROMA FUNCTIONS ###############


"""""""""""""""""""""""""""""""""""""""""""""""""""
The pick_up_tray function will take a plate location
as an argument and pick up the tray at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def pick_up_tray(plate):
  await backend.send_command("C1", "PAZ", [z_travel])
  if plate == carousel:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_carousel])
  elif plate == sealer:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_sealer])
  elif plate == scanner:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_regrip, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_scanner])
  else:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_plate, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_tray])
  await backend.send_command("C1", "PAG", [g_tray_closed])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The transfer_tray_to function will take a plate location
as an argument and place a held tray at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def transfer_tray_to(plate):
  await backend.send_command("C1", "PAZ", [z_travel])
  if plate == carousel:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, None])
    await backend.send_command("C1", "PAZ", [z_carousel])
  elif plate == sealer:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, None])
    await backend.send_command("C1", "PAZ", [z_sealer])
  elif plate == scanner:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_regrip, None])
    await backend.send_command("C1", "PAZ", [z_scanner])
  else:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_plate, None])
    await backend.send_command("C1", "PAZ", [z_tray])
  await backend.send_command("C1", "PAG", [g_tray_open])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_tray function will take 2 arguments.
The first argument is a pickup location,
the second is where the tray will be deposited
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_tray(initial_location, end_location):
  await pick_up_tray(initial_location)
  await transfer_tray_to(end_location)

"""""""""""""""""""""""""""""""""""""""""""""""""""
The pick_up_bucket function will take a bucket location
as an argument and pick up the bucket at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def pick_up_bucket(bucket):
  await backend.send_command("C1", "PAZ", [z_travel])
  if bucket == regrip:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_regrip, g_bucket_open_thin])
  else:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_plate, g_bucket_open_thin])
  await backend.send_command("C1", "PAZ", [z_bucket_thin])
  await backend.send_command("C1", "PAG", [g_bucket_closed_thin])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The transfer_bucket_to function will take a bucket location
as an argument and place a held bucket at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def transfer_bucket_to(bucket):
  await backend.send_command("C1", "PAZ", [z_travel])
  if bucket == regrip:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_regrip, None])
  else:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_plate, None])
  await backend.send_command("C1", "PAZ", [z_bucket_thin])
  await backend.send_command("C1", "PAG", [g_bucket_open_thin])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_bucket function will take 2 arguments.
The first argument is a pickup location,
the second is where the tray will be deposited
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_bucket(initial_location, end_location):
  pick_up_bucket(initial_location)
  transfer_bucket_to(end_location)

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_into_centrifuge function will transfer a bucket
from the regrip plate into the centrifuge
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_into_centrifuge():
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [regrip_wide.x, regrip_wide.y, None, r_plate, g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_bucket_wide])
  await backend.send_command("C1", "PAG", [g_bucket_closed_wide])
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [centrifuge.x, centrifuge.y, None, r_regrip, None])
  await backend.send_command("C1", "PAZ", [z_centrifuge])
  await backend.send_command("C1", "PAG", [g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_from_centrifuge function will transfer a bucket
from the centrifuge onto the regrip plate
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_from_centrifuge():
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [centrifuge.x, centrifuge.y, None, r_regrip, g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_centrifuge])
  await backend.send_command("C1", "PAG", [g_bucket_closed_wide])
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [regrip_wide.x, regrip_wide.y, None, r_plate, None])
  await backend.send_command("C1", "PAZ", [z_bucket_wide])
  await backend.send_command("C1", "PAG", [g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_travel])


In [ ]:
############### LIHA FUNCTIONS ###############

In [ ]:
############### PNP FUNCTIONS ###############

In [ ]:
############### TESTING CELL ###############

async def test():
  backend = EVOBackend()
  await backend.io.setup()
  resp1 = await backend.send_command("C1", "PIA")
  print(resp1)

  await backend.send_command("C1", "PAA", [14628, 1999, 2000, 1800, 900])
  await backend.send_command("C1", "PAR", [r_tray])
  await backend.send_command("C1", "PAX", [plate_1.x])
  await backend.send_command("C1", "PAY", [plate_1.y])
  await backend.send_command("C1", "PAZ", [z_tray])
  await backend.send_command("C1", "PAG", [g_tray_closed])
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAY", [1330])
  await backend.send_command("C1", "PAZ", [z_tray])
  await backend.send_command("C1", "PAG", [g_tray_open])
  await backend.send_command("C1", "PAZ", [z_travel])

  await backend.stop

await test()